In [1]:
# ==========================================
# STEP 9.2: IMPORT LIBRARIES
# ==========================================

import os
import re
import unicodedata
import pandas as pd
import numpy as np

from collections import defaultdict

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# ==========================================
# STEP 9.3: SET PATHS
# ==========================================

TRAIN_DIR = r"C:\Users\anupr\Amazon ML Challenge\Train"

OUTPUT_DIR = r"C:\Users\anupr\Amazon ML Challenge\Output"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Training directory:", TRAIN_DIR)
print("Output directory:", OUTPUT_DIR)

print("\nTrain directory exists:",
      os.path.exists(TRAIN_DIR))

print("Output directory exists:",
      os.path.exists(OUTPUT_DIR))

Training directory: C:\Users\anupr\Amazon ML Challenge\Train
Output directory: C:\Users\anupr\Amazon ML Challenge\Output

Train directory exists: True
Output directory exists: True


In [3]:
# ==========================================
# STEP 9.4: LOAD DATA
# ==========================================

print("Loading Source 1...")

s1 = pd.read_parquet(
    os.path.join(
        TRAIN_DIR,
        "train_source1.parquet"
    )
)

print("Loading Source 2...")

s2 = pd.read_parquet(
    os.path.join(
        TRAIN_DIR,
        "train_source2.parquet"
    )
)
print("Loading Source 3...")

s3 = pd.read_parquet(
    os.path.join(
        TRAIN_DIR,
        "train_source3.parquet"
    )
)

print("\nDatasets loaded successfully!")

print("Source 1:", s1.shape)
print("Source 2:", s2.shape)
print("Source 3:", s3.shape)

Loading Source 1...
Loading Source 2...
Loading Source 3...

Datasets loaded successfully!
Source 1: (2206821, 4)
Source 2: (5034616, 4)
Source 3: (5285603, 4)


In [4]:
# ==========================================
# STEP 9.5: CHECK NORMALIZED COLUMNS
# ==========================================

print("Source 1 columns:")
print(s1.columns.tolist())

print("\nSource 2 columns:")
print(s2.columns.tolist())

print("\nSource 3 columns:")
print(s3.columns.tolist())

Source 1 columns:
['entity_id', 'business_name', 'business_address', 'country']

Source 2 columns:
['entity_id', 'business_name', 'business_address', 'country']

Source 3 columns:
['entity_id', 'business_name', 'business_address', 'country']


In [5]:
# ==========================================
# STEP 9.6: NORMALIZATION FUNCTIONS
# ==========================================

def normalize_text(text):

    if pd.isna(text):
        return ""

    text = str(text).strip().lower()

    text = unicodedata.normalize(
        "NFKC",
        text
    )

    text = re.sub(
        r"[^\w\s]",
        " ",
        text,
        flags=re.UNICODE
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


def normalize_name(text):

    text = normalize_text(text)

    replacements = {
        "pvt": "private",
        "ltd": "limited",
        "corp": "corporation",
        "co": "company",
        "inc": "incorporated"
    }

    words = text.split()

    words = [
        replacements.get(word, word)
        for word in words
    ]

    return " ".join(words)


def normalize_address(text):

    text = normalize_text(text)

    replacements = {
        "rd": "road",
        "st": "street",
        "ave": "avenue",
        "av": "avenue",
        "blvd": "boulevard",
        "ln": "lane",
        "dr": "drive",
        "hwy": "highway"
    }

    words = text.split()

    words = [
        replacements.get(word, word)
        for word in words
    ]

    return " ".join(words)

In [6]:
# ==========================================
# APPLY NORMALIZATION
# ==========================================

for df in [s1, s2, s3]:

    df["name_norm"] = (
        df["business_name"]
        .apply(normalize_name)
    )

    df["address_norm"] = (
        df["business_address"]
        .apply(normalize_address)
    )

    df["country_norm"] = (
        df["country"]
        .fillna("")
        .astype(str)
        .str.lower()
        .str.strip()
    )

print("Normalization completed!")

Normalization completed!


In [7]:
# ==========================================
# STEP 9.7: TOKEN FUNCTIONS
# ==========================================

def get_name_tokens(name):

    if not name:
        return []

    return [
        token
        for token in name.split()
        if len(token) >= 3
    ]


def get_address_tokens(address):

    if not address:
        return []

    return [
        token
        for token in address.split()
        if len(token) >= 3
    ]

print("Token functions ready!")

Token functions ready!


In [8]:
# ==========================================
# STEP 9.8: BUILD BLOCKING INDEXES
# ==========================================

def build_token_index(
    df,
    column,
    tokenizer
):

    index = defaultdict(list)

    for entity_id, text, country in zip(
        df["entity_id"],
        df[column],
        df["country_norm"]
    ):

        tokens = tokenizer(text)

        for token in set(tokens):

            index[
                (country, token)
            ].append(entity_id)

    return index

In [9]:
print("Building S2 name index...")

s2_name_index = build_token_index(
    s2,
    "name_norm",
    get_name_tokens
)

print("Building S3 name index...")

s3_name_index = build_token_index(
    s3,
    "name_norm",
    get_name_tokens
)

Building S2 name index...
Building S3 name index...


In [10]:
print("Building S2 address index...")

s2_address_index = build_token_index(
    s2,
    "address_norm",
    get_address_tokens
)

print("Building S3 address index...")

s3_address_index = build_token_index(
    s3,
    "address_norm",
    get_address_tokens
)

print("\nAll blocking indexes created!")

Building S2 address index...
Building S3 address index...

All blocking indexes created!


In [11]:
# ==========================================
# STEP 9.9: CANDIDATE GENERATOR
# ==========================================

def generate_candidates(row):

    country = row["country_norm"]

    name_tokens = set(
        get_name_tokens(
            row["name_norm"]
        )
    )

    address_tokens = set(
        get_address_tokens(
            row["address_norm"]
        )
    )

    candidates = set()

    # ------------------------------
    # NAME BLOCKING
    # ------------------------------

    for token in name_tokens:

        candidates.update(
            s2_name_index.get(
                (country, token),
                []
            )
        )

        candidates.update(
            s3_name_index.get(
                (country, token),
                []
            )
        )

    # ------------------------------
    # ADDRESS BLOCKING
    # ------------------------------

    for token in address_tokens:

        candidates.update(
            s2_address_index.get(
                (country, token),
                []
            )
        )

        candidates.update(
            s3_address_index.get(
                (country, token),
                []
            )
        )

    return candidates

In [12]:
# ==========================================
# STEP 9.10: TEST CANDIDATE GENERATION
# ==========================================

test_sample = s1.sample(
    n=100,
    random_state=42
).copy()

candidate_counts = []

for _, row in test_sample.iterrows():

    candidates = generate_candidates(row)

    candidate_counts.append(
        len(candidates)
    )

print("Candidate generation test complete!")

print("\nCandidate statistics:")
print(
    pd.Series(candidate_counts).describe()
)

Candidate generation test complete!

Candidate statistics:
count    1.000000e+02
mean     1.654152e+06
std      7.681237e+05
min      4.870000e+02
25%      1.119868e+06
50%      1.719511e+06
75%      2.284286e+06
max      2.974697e+06
dtype: float64


In [13]:
# ============================================
# STEP 9.11: ANALYZE HIGH-FREQUENCY TOKENS
# ============================================

from collections import Counter

def get_index_statistics(index, index_name):
    frequencies = []

    for (country, token), entity_ids in index.items():
        frequencies.append({
            "country": country,
            "token": token,
            "frequency": len(entity_ids)
        })

    df_freq = pd.DataFrame(frequencies)

    print(f"\n{index_name}")
    print("-" * 50)
    print("Total unique blocks:", len(df_freq))
    print("Maximum block size:", df_freq["frequency"].max())
    print("Mean block size:", round(df_freq["frequency"].mean(), 2))
    print("Median block size:", round(df_freq["frequency"].median(), 2))

    print("\nTop 20 most frequent tokens:")
    display(
        df_freq.sort_values("frequency", ascending=False)
        .head(20)
    )

    return df_freq


s2_name_freq = get_index_statistics(
    s2_name_index,
    "Source 2 - Name Token Index"
)

s3_name_freq = get_index_statistics(
    s3_name_index,
    "Source 3 - Name Token Index"
)

s2_address_freq = get_index_statistics(
    s2_address_index,
    "Source 2 - Address Token Index"
)

s3_address_freq = get_index_statistics(
    s3_address_index,
    "Source 3 - Address Token Index"
)


Source 2 - Name Token Index
--------------------------------------------------
Total unique blocks: 838931
Maximum block size: 834466
Mean block size: 17.96
Median block size: 1.0

Top 20 most frequent tokens:


,country,token,frequency
19,india,limited,834466
18,india,private,716649
21,us,llc,528290
3,us,incorporated,426416
120,us,corporation,163446
110,us,center,137522
29,us,com,132650
377,us,partners,129317
10,us,and,115563
158,india,india,110684



Source 3 - Name Token Index
--------------------------------------------------
Total unique blocks: 900475
Maximum block size: 1013601
Mean block size: 18.57
Median block size: 1.0

Top 20 most frequent tokens:


,country,token,frequency
2,india,limited,1013601
5,india,private,859372
10,us,llc,571331
24,us,incorporated,441683
95,us,corporation,157672
9,us,center,147726
1,us,com,133754
20,us,partners,133679
318,us,and,119531
65,india,india,112959



Source 2 - Address Token Index
--------------------------------------------------
Total unique blocks: 639019
Maximum block size: 585348
Mean block size: 44.21
Median block size: 2.0

Top 20 most frequent tokens:


,country,token,frequency
5,us,street,585348
51,us,road,545588
40,us,drive,487074
25,india,road,453182
20,us,avenue,401290
29,india,floor,319584
42,india,maharashtra,319026
45,india,nagar,286965
0,india,delhi,262208
12,india,pradesh,214708



Source 3 - Address Token Index
--------------------------------------------------
Total unique blocks: 611431
Maximum block size: 617747
Mean block size: 50.05
Median block size: 2.0

Top 20 most frequent tokens:


,country,token,frequency
76,us,street,617747
2,us,road,574872
59,us,drive,512945
12,us,avenue,422965
103,india,road,369277
4,us,texas,293309
78,us,new,275077
118,india,delhi,272632
104,india,nagar,267813
99,india,floor,264672


In [15]:
# ============================================
# LOAD GROUND TRUTH
# ============================================

gt = pd.read_parquet(
    os.path.join(TRAIN_DIR, "train_ground_truth.parquet")
)

print("Ground truth loaded successfully!")
print("Ground truth shape:", gt.shape)
print("Ground truth columns:", gt.columns.tolist())

Ground truth loaded successfully!
Ground truth shape: (2206821, 2)
Ground truth columns: ['source1_entity_id', 'matched_entity_ids']


In [17]:
# ============================================
# PREPARE EVALUATION SAMPLE
# ============================================

# Sample actual Source 1 records so that
# name_norm, address_norm and country_norm
# are available for candidate generation.

sample_s1 = s1.sample(
    n=5000,
    random_state=42
).copy()

# Attach ground-truth matches
sample_gt = sample_s1.merge(
    gt[["source1_entity_id", "matched_entity_ids"]],
    left_on="entity_id",
    right_on="source1_entity_id",
    how="left"
)

print("Evaluation sample prepared!")
print("Sample size:", len(sample_gt))

print("\nColumns available:")
print(sample_gt.columns.tolist())

Evaluation sample prepared!
Sample size: 5000

Columns available:
['entity_id', 'business_name', 'business_address', 'country', 'name_norm', 'address_norm', 'country_norm', 'source1_entity_id', 'matched_entity_ids']


In [18]:
print(sample_gt[
    [
        "entity_id",
        "country_norm",
        "name_norm",
        "address_norm",
        "matched_entity_ids"
    ]
].head())

      entity_id country_norm                                  name_norm  \
0   S1-53356671           us                    pediatric medicine pllc   
1  S1-320151505           us                       fetech national twin   
2  S1-938947364           us             general design innovations llc   
3  S1-195839862        india  construction ideaz papers private limited   
4  S1-655046555           us     penaloza and bittle first incorporated   

                                        address_norm  \
0                                  4850 20 otisco ny   
1                    19034 woodburn road woodburn in   
2                          7241 osage avenue mesa az   
3  building no 4 606 prabhul cottage karimbalur e...   
4                 4255 charleswood avenue memphis tn   

                                  matched_entity_ids  
0  S2-453334141,S2-904412198,S2-256033433,S3-4686...  
1             S2-230628758,S2-951553277,S3-975397820  
2             S2-159830603,S3-544960509,S3-2680

In [20]:
# ============================================
# STEP 9.12A: PREPARE FAST RECALL EVALUATION
# ============================================

# Sample Source 1 records
sample_s1 = s1.sample(
    n=5000,
    random_state=42
).copy()

# Attach ground truth
sample_gt = sample_s1.merge(
    gt[["source1_entity_id", "matched_entity_ids"]],
    left_on="entity_id",
    right_on="source1_entity_id",
    how="left"
)

# --------------------------------------------
# Create fast entity lookups
# --------------------------------------------

s2_lookup = (
    s2.set_index("entity_id")[
        ["name_norm", "address_norm", "country_norm"]
    ]
    .to_dict("index")
)

s3_lookup = (
    s3.set_index("entity_id")[
        ["name_norm", "address_norm", "country_norm"]
    ]
    .to_dict("index")
)

s2_ids = set(s2_lookup.keys())
s3_ids = set(s3_lookup.keys())


# --------------------------------------------
# Parse ground-truth matches
# --------------------------------------------

def parse_matches(value):

    if pd.isna(value):
        return set()

    value = str(value).strip()

    if not value:
        return set()

    return set(
        x.strip()
        for x in value.split(",")
        if x.strip()
    )


# --------------------------------------------
# Build token frequency dictionaries
# --------------------------------------------

def get_token_frequency(index):

    return {
        key: len(entity_ids)
        for key, entity_ids in index.items()
    }


s2_name_freq = get_token_frequency(s2_name_index)
s3_name_freq = get_token_frequency(s3_name_index)

s2_address_freq = get_token_frequency(s2_address_index)
s3_address_freq = get_token_frequency(s3_address_index)


print("Fast recall evaluation setup complete!")

print("\nEvaluation records:", len(sample_gt))
print("S2 entities:", len(s2_ids))
print("S3 entities:", len(s3_ids))

Fast recall evaluation setup complete!

Evaluation records: 5000
S2 entities: 5034616
S3 entities: 5285603


In [21]:
# ============================================
# STEP 9.12B: FAST FREQUENCY THRESHOLD TEST
# ============================================

thresholds = [25000, 50000, 100000, 200000, 500000]


def is_match_covered(
    source1_row,
    target_id,
    threshold
):

    country = source1_row["country_norm"]

    source1_name_tokens = set(
        get_name_tokens(source1_row["name_norm"])
    )

    source1_address_tokens = set(
        get_address_tokens(source1_row["address_norm"])
    )

    # ----------------------------------------
    # Identify target source
    # ----------------------------------------

    if target_id in s2_lookup:
        target = s2_lookup[target_id]

        target_name_index = s2_name_freq
        target_address_index = s2_address_freq

    elif target_id in s3_lookup:
        target = s3_lookup[target_id]

        target_name_index = s3_name_freq
        target_address_index = s3_address_freq

    else:
        return False

    # ----------------------------------------
    # Country must match
    # ----------------------------------------

    if country != target["country_norm"]:
        return False

    # ----------------------------------------
    # Name-token blocking
    # ----------------------------------------

    target_name_tokens = set(
        get_name_tokens(target["name_norm"])
    )

    shared_name_tokens = (
        source1_name_tokens &
        target_name_tokens
    )

    for token in shared_name_tokens:

        frequency = target_name_index.get(
            (country, token),
            0
        )

        if frequency <= threshold:
            return True

    # ----------------------------------------
    # Address-token blocking
    # ----------------------------------------

    target_address_tokens = set(
        get_address_tokens(target["address_norm"])
    )

    shared_address_tokens = (
        source1_address_tokens &
        target_address_tokens
    )

    for token in shared_address_tokens:

        frequency = target_address_index.get(
            (country, token),
            0
        )

        if frequency <= threshold:
            return True

    return False


# --------------------------------------------
# Evaluate thresholds
# --------------------------------------------

results = []

for threshold in thresholds:

    print(f"\nTesting threshold: {threshold}")

    total_actual_matches = 0
    covered_matches = 0

    for _, row in sample_gt.iterrows():

        actual_matches = parse_matches(
            row["matched_entity_ids"]
        )

        for target_id in actual_matches:

            total_actual_matches += 1

            if is_match_covered(
                row,
                target_id,
                threshold
            ):
                covered_matches += 1

    recall = (
        covered_matches / total_actual_matches
        if total_actual_matches
        else 0
    )

    results.append({
        "threshold": threshold,
        "total_actual_matches": total_actual_matches,
        "covered_matches": covered_matches,
        "missed_matches": (
            total_actual_matches - covered_matches
        ),
        "recall_percent": recall * 100
    })

    print(
        f"Recall: {recall * 100:.2f}%"
    )


results_df = pd.DataFrame(results)

print("\n" + "=" * 80)
print("FREQUENCY THRESHOLD RECALL COMPARISON")
print("=" * 80)

display(
    results_df.round(2)
)


Testing threshold: 25000
Recall: 99.26%

Testing threshold: 50000
Recall: 99.45%

Testing threshold: 100000
Recall: 99.70%

Testing threshold: 200000
Recall: 99.91%

Testing threshold: 500000
Recall: 99.93%

FREQUENCY THRESHOLD RECALL COMPARISON


,threshold,total_actual_matches,covered_matches,missed_matches,recall_percent
0,25000,17383,17255,128,99.26
1,50000,17383,17287,96,99.45
2,100000,17383,17330,53,99.70
3,200000,17383,17367,16,99.91
4,500000,17383,17371,12,99.93


In [22]:
# ============================================
# STEP 9.13: BUILD FINAL FILTERED INDEXES
# ============================================

MAX_BLOCK_FREQUENCY = 200_000


def filter_index_by_frequency(index, max_frequency):
    return {
        key: entity_ids
        for key, entity_ids in index.items()
        if len(entity_ids) <= max_frequency
    }


# Filter name indexes
s2_name_filtered = filter_index_by_frequency(
    s2_name_index,
    MAX_BLOCK_FREQUENCY
)

s3_name_filtered = filter_index_by_frequency(
    s3_name_index,
    MAX_BLOCK_FREQUENCY
)


# Filter address indexes
s2_address_filtered = filter_index_by_frequency(
    s2_address_index,
    MAX_BLOCK_FREQUENCY
)

s3_address_filtered = filter_index_by_frequency(
    s3_address_index,
    MAX_BLOCK_FREQUENCY
)


print("Final blocking indexes created!")
print()
print("Maximum block frequency:", MAX_BLOCK_FREQUENCY)

print("\nName indexes:")
print("S2:", len(s2_name_filtered))
print("S3:", len(s3_name_filtered))

print("\nAddress indexes:")
print("S2:", len(s2_address_filtered))
print("S3:", len(s3_address_filtered))

Final blocking indexes created!

Maximum block frequency: 200000

Name indexes:
S2: 838927
S3: 900471

Address indexes:
S2: 639007
S3: 611416


In [23]:
# ============================================
# STEP 9.14: TEST FINAL CANDIDATE SIZE
# ============================================

def generate_filtered_candidates(row):

    country = row["country_norm"]

    name_tokens = set(
        get_name_tokens(row["name_norm"])
    )

    address_tokens = set(
        get_address_tokens(row["address_norm"])
    )

    candidates = set()

    # -------------------------------
    # Name candidates
    # -------------------------------

    for token in name_tokens:

        candidates.update(
            s2_name_filtered.get(
                (country, token),
                []
            )
        )

        candidates.update(
            s3_name_filtered.get(
                (country, token),
                []
            )
        )

    # -------------------------------
    # Address candidates
    # -------------------------------

    for token in address_tokens:

        candidates.update(
            s2_address_filtered.get(
                (country, token),
                []
            )
        )

        candidates.update(
            s3_address_filtered.get(
                (country, token),
                []
            )
        )

    return candidates


# Test only 100 Source-1 records
test_sample = s1.sample(
    n=100,
    random_state=42
).copy()


candidate_counts = []

for _, row in test_sample.iterrows():

    candidates = generate_filtered_candidates(row)

    candidate_counts.append(
        len(candidates)
    )


candidate_stats = pd.Series(
    candidate_counts
).describe()


print("Filtered candidate generation test complete!")

print("\nCandidate statistics:")
print(candidate_stats)

Filtered candidate generation test complete!

Candidate statistics:
count       100.000000
mean     272815.770000
std      219429.684546
min         487.000000
25%       79918.000000
50%      229197.000000
75%      419328.500000
max      911239.000000
dtype: float64


In [24]:
# ============================================
# STEP 9.15: COMPARE CANDIDATE VOLUME
# ============================================

candidate_thresholds = [
    25000,
    50000,
    100000,
    200000
]


def build_filtered_indexes(threshold):

    return (
        filter_index_by_frequency(
            s2_name_index,
            threshold
        ),
        filter_index_by_frequency(
            s3_name_index,
            threshold
        ),
        filter_index_by_frequency(
            s2_address_index,
            threshold
        ),
        filter_index_by_frequency(
            s3_address_index,
            threshold
        )
    )


def generate_candidates_with_indexes(
    row,
    s2_name_idx,
    s3_name_idx,
    s2_address_idx,
    s3_address_idx
):

    country = row["country_norm"]

    name_tokens = set(
        get_name_tokens(row["name_norm"])
    )

    address_tokens = set(
        get_address_tokens(row["address_norm"])
    )

    candidates = set()

    # Name blocks
    for token in name_tokens:

        candidates.update(
            s2_name_idx.get(
                (country, token),
                []
            )
        )

        candidates.update(
            s3_name_idx.get(
                (country, token),
                []
            )
        )

    # Address blocks
    for token in address_tokens:

        candidates.update(
            s2_address_idx.get(
                (country, token),
                []
            )
        )

        candidates.update(
            s3_address_idx.get(
                (country, token),
                []
            )
        )

    return candidates


volume_results = []

# Use the same 100-record test sample
test_sample = s1.sample(
    n=100,
    random_state=42
).copy()


for threshold in candidate_thresholds:

    print(
        f"\nTesting candidate threshold: {threshold}"
    )

    (
        s2_name_idx,
        s3_name_idx,
        s2_address_idx,
        s3_address_idx
    ) = build_filtered_indexes(threshold)

    counts = []

    for _, row in test_sample.iterrows():

        candidates = generate_candidates_with_indexes(
            row,
            s2_name_idx,
            s3_name_idx,
            s2_address_idx,
            s3_address_idx
        )

        counts.append(len(candidates))

    stats = pd.Series(counts)

    volume_results.append({
        "threshold": threshold,
        "mean_candidates": stats.mean(),
        "median_candidates": stats.median(),
        "p75_candidates": stats.quantile(0.75),
        "max_candidates": stats.max()
    })

    print(
        f"Mean:   {stats.mean():,.0f}"
    )
    print(
        f"Median: {stats.median():,.0f}"
    )
    print(
        f"Max:    {stats.max():,.0f}"
    )


volume_df = pd.DataFrame(volume_results)

print("\n" + "=" * 80)
print("CANDIDATE VOLUME COMPARISON")
print("=" * 80)

display(
    volume_df.round(0)
)


Testing candidate threshold: 25000
Mean:   57,519
Median: 51,430
Max:    156,302

Testing candidate threshold: 50000
Mean:   96,185
Median: 67,863
Max:    433,727

Testing candidate threshold: 100000
Mean:   167,808
Median: 136,302
Max:    559,990

Testing candidate threshold: 200000
Mean:   272,816
Median: 229,197
Max:    911,239

CANDIDATE VOLUME COMPARISON


,threshold,mean_candidates,median_candidates,p75_candidates,max_candidates
0,25000,57519.0,51430.0,82263.0,156302
1,50000,96185.0,67863.0,136955.0,433727
2,100000,167808.0,136302.0,268478.0,559990
3,200000,272816.0,229197.0,419328.0,911239


In [25]:
# ============================================
# STEP 9.16: ANALYZE TOKEN COUNTS
# ============================================

name_token_counts = s1["name_norm"].apply(
    lambda x: len(set(get_name_tokens(x)))
)

address_token_counts = s1["address_norm"].apply(
    lambda x: len(set(get_address_tokens(x)))
)

print("NAME TOKEN COUNTS")
print("=" * 50)
print(name_token_counts.describe())

print("\n\nADDRESS TOKEN COUNTS")
print("=" * 50)
print(address_token_counts.describe())

NAME TOKEN COUNTS
count    2.206821e+06
mean     3.333429e+00
std      9.645738e-01
min      0.000000e+00
25%      3.000000e+00
50%      3.000000e+00
75%      4.000000e+00
max      1.500000e+01
Name: name_norm, dtype: float64


ADDRESS TOKEN COUNTS
count    2.206821e+06
mean     6.325676e+00
std      2.878270e+00
min      1.000000e+00
25%      4.000000e+00
50%      5.000000e+00
75%      8.000000e+00
max      3.000000e+01
Name: address_norm, dtype: float64


In [26]:
# ============================================
# STEP 9.16B: SAMPLE TOKEN DISTRIBUTION
# ============================================

print("Example Source 1 names and tokens:")
print("=" * 80)

for _, row in s1.sample(
    n=10,
    random_state=42
).iterrows():

    print("\nBusiness:", row["business_name"])
    print("Tokens:", get_name_tokens(row["name_norm"]))

    print("Address:", row["business_address"])
    print("Tokens:", get_address_tokens(row["address_norm"]))

Example Source 1 names and tokens:

Business: Pediatric Medicine PLLC
Tokens: ['pediatric', 'medicine', 'pllc']
Address: 4850 20, Otisco, NY
Tokens: ['4850', 'otisco']

Business: Fetech National Twin
Tokens: ['fetech', 'national', 'twin']
Address: 19034 Woodburn Road, Woodburn, IN
Tokens: ['19034', 'woodburn', 'road', 'woodburn']

Business: General Design Innovations LLC
Tokens: ['general', 'design', 'innovations', 'llc']
Address: 7241 Osage Avenue, Mesa, AZ
Tokens: ['7241', 'osage', 'avenue', 'mesa']

Business: Construction Ideaz Papers Private Limited
Tokens: ['construction', 'ideaz', 'papers', 'private', 'limited']
Address: Building No.4/606, Prabhul Cottage Karimbalur, Elakamon, Ayiroor P.O., Thiruvananthapuram, Trivandrum, Kerala
Tokens: ['building', '606', 'prabhul', 'cottage', 'karimbalur', 'elakamon', 'ayiroor', 'thiruvananthapuram', 'trivandrum', 'kerala']

Business: Penaloza and Bittle First Inc.
Tokens: ['penaloza', 'and', 'bittle', 'first', 'incorporated']
Address: 4255 Cha

In [27]:
# ============================================
# STEP 9.17: RAREST-TOKEN BLOCKING
# ============================================

RAREST_TOKEN_THRESHOLD = 100_000


def get_rare_tokens(
    tokens,
    country,
    frequency_index,
    max_frequency=100_000
):
    """
    Return tokens whose country-specific block
    frequency is <= max_frequency.
    """

    rare_tokens = []

    for token in set(tokens):

        frequency = frequency_index.get(
            (country, token),
            0
        )

        if frequency > 0 and frequency <= max_frequency:
            rare_tokens.append(
                (token, frequency)
            )

    # Rarest tokens first
    rare_tokens.sort(
        key=lambda x: x[1]
    )

    return rare_tokens


def generate_rare_token_candidates(row):

    country = row["country_norm"]

    # ----------------------------------------
    # Name tokens
    # ----------------------------------------

    name_tokens = get_name_tokens(
        row["name_norm"]
    )

    name_candidates = []

    # Check S2 + S3 frequencies together
    for token in set(name_tokens):

        s2_freq = s2_name_freq.get(
            (country, token),
            0
        )

        s3_freq = s3_name_freq.get(
            (country, token),
            0
        )

        frequencies = [
            f for f in [s2_freq, s3_freq]
            if f > 0
        ]

        if frequencies:

            frequency = min(frequencies)

            if frequency <= RAREST_TOKEN_THRESHOLD:
                name_candidates.append(
                    (token, frequency)
                )

    name_candidates.sort(
        key=lambda x: x[1]
    )


    # ----------------------------------------
    # Address tokens
    # ----------------------------------------

    address_tokens = get_address_tokens(
        row["address_norm"]
    )

    address_candidates = []

    for token in set(address_tokens):

        s2_freq = s2_address_freq.get(
            (country, token),
            0
        )

        s3_freq = s3_address_freq.get(
            (country, token),
            0
        )

        frequencies = [
            f for f in [s2_freq, s3_freq]
            if f > 0
        ]

        if frequencies:

            frequency = min(frequencies)

            if frequency <= RAREST_TOKEN_THRESHOLD:
                address_candidates.append(
                    (token, frequency)
                )

    address_candidates.sort(
        key=lambda x: x[1]
    )


    # ----------------------------------------
    # Use the two rarest tokens
    # ----------------------------------------

    selected_name = name_candidates[:2]
    selected_address = address_candidates[:2]


    candidates = set()

    # Name
    for token, _ in selected_name:

        candidates.update(
            s2_name_filtered.get(
                (country, token),
                []
            )
        )

        candidates.update(
            s3_name_filtered.get(
                (country, token),
                []
            )
        )

    # Address
    for token, _ in selected_address:

        candidates.update(
            s2_address_filtered.get(
                (country, token),
                []
            )
        )

        candidates.update(
            s3_address_filtered.get(
                (country, token),
                []
            )
        )

    return candidates

In [28]:
# ============================================
# STEP 9.17B: BUILD 100K FILTERED INDEXES
# ============================================

RAREST_TOKEN_THRESHOLD = 100_000

s2_name_filtered = filter_index_by_frequency(
    s2_name_index,
    RAREST_TOKEN_THRESHOLD
)

s3_name_filtered = filter_index_by_frequency(
    s3_name_index,
    RAREST_TOKEN_THRESHOLD
)

s2_address_filtered = filter_index_by_frequency(
    s2_address_index,
    RAREST_TOKEN_THRESHOLD
)

s3_address_filtered = filter_index_by_frequency(
    s3_address_index,
    RAREST_TOKEN_THRESHOLD
)

print("Rare-token indexes ready.")
print("Threshold:", RAREST_TOKEN_THRESHOLD)

Rare-token indexes ready.
Threshold: 100000


In [29]:
# ============================================
# STEP 9.18: TEST RAREST-TOKEN CANDIDATE SIZE
# ============================================

test_sample = s1.sample(
    n=100,
    random_state=42
).copy()

candidate_counts = []

for _, row in test_sample.iterrows():

    candidates = generate_rare_token_candidates(row)

    candidate_counts.append(
        len(candidates)
    )

stats = pd.Series(candidate_counts).describe()

print("Rare-token candidate generation test complete!")

print("\nCandidate statistics:")
print(stats)

Rare-token candidate generation test complete!

Candidate statistics:
count       100.000000
mean      30565.730000
std       35306.909925
min         116.000000
25%        5585.250000
50%       22021.500000
75%       37453.500000
max      178320.000000
dtype: float64


In [30]:
# ============================================
# STEP 9.19: COMPARE RAREST TOKEN COUNTS
# ============================================

def generate_rare_token_candidates_config(
    row,
    max_frequency,
    max_tokens_per_field
):
    """
    Generate candidates using the N rarest
    eligible name tokens and address tokens.
    """

    country = row["country_norm"]

    candidates = set()

    # ----------------------------------------
    # NAME TOKENS
    # ----------------------------------------

    name_candidates = []

    for token in set(
        get_name_tokens(row["name_norm"])
    ):

        s2_freq = s2_name_freq.get(
            (country, token),
            0
        )

        s3_freq = s3_name_freq.get(
            (country, token),
            0
        )

        frequencies = [
            f for f in [s2_freq, s3_freq]
            if f > 0
        ]

        if frequencies:

            # Use the smaller available block
            frequency = min(frequencies)

            if frequency <= max_frequency:
                name_candidates.append(
                    (token, frequency)
                )

    name_candidates.sort(
        key=lambda x: x[1]
    )

    # ----------------------------------------
    # ADDRESS TOKENS
    # ----------------------------------------

    address_candidates = []

    for token in set(
        get_address_tokens(row["address_norm"])
    ):

        s2_freq = s2_address_freq.get(
            (country, token),
            0
        )

        s3_freq = s3_address_freq.get(
            (country, token),
            0
        )

        frequencies = [
            f for f in [s2_freq, s3_freq]
            if f > 0
        ]

        if frequencies:

            frequency = min(frequencies)

            if frequency <= max_frequency:
                address_candidates.append(
                    (token, frequency)
                )

    address_candidates.sort(
        key=lambda x: x[1]
    )

    # ----------------------------------------
    # SELECT N RAREST
    # ----------------------------------------

    selected_name = name_candidates[
        :max_tokens_per_field
    ]

    selected_address = address_candidates[
        :max_tokens_per_field
    ]

    # ----------------------------------------
    # RETRIEVE CANDIDATES
    # ----------------------------------------

    for token, _ in selected_name:
        candidates.update(
            s2_name_filtered.get(
                (country, token),
                []
            )
        )
        candidates.update(
            s3_name_filtered.get(
                (country, token),
                []
            )
        )
    for token, _ in selected_address:
        candidates.update(
            s2_address_filtered.get(
                (country, token),
                []
            )
        )
        candidates.update(
            s3_address_filtered.get(
                (country, token),
                []
            )
        )
    return candidates
# --------------------------------------------
# Test different configurations
# --------------------------------------------

configs = [
    (25000, 1),
    (25000, 2),
    (50000, 1),
    (50000, 2),
    (100000, 1),
    (100000, 2)
]
test_sample = s1.sample(
    n=100,
    random_state=42
).copy()
volume_results = []
for max_frequency, n_tokens in configs:
    print(
        f"\nTesting frequency={max_frequency}, "
        f"tokens={n_tokens}"
    )
    counts = []
    for _, row in test_sample.iterrows():

        candidates = generate_rare_token_candidates_config(
            row,
            max_frequency,
            n_tokens
        )

        counts.append(len(candidates))
    stats = pd.Series(counts)
    volume_results.append({
        "max_frequency": max_frequency,
        "tokens_per_field": n_tokens,
        "mean_candidates": stats.mean(),
        "median_candidates": stats.median(),
        "p75_candidates": stats.quantile(0.75),
        "max_candidates": stats.max()
    })
volume_df = pd.DataFrame(volume_results)
print("\n" + "=" * 80)
print("RAREST-TOKEN CANDIDATE VOLUME COMPARISON")
print("=" * 80)
display(
    volume_df.round(0)
)


Testing frequency=25000, tokens=1

Testing frequency=25000, tokens=2

Testing frequency=50000, tokens=1

Testing frequency=50000, tokens=2

Testing frequency=100000, tokens=1

Testing frequency=100000, tokens=2

RAREST-TOKEN CANDIDATE VOLUME COMPARISON


,max_frequency,tokens_per_field,mean_candidates,median_candidates,p75_candidates,max_candidates
0,25000,1,6916.0,1818.0,9710.0,49997
1,25000,2,20400.0,16460.0,32233.0,81252
2,50000,1,7721.0,1884.0,9730.0,80781
3,50000,2,26127.0,20874.0,35734.0,128050
4,100000,1,7721.0,1884.0,9730.0,80781
5,100000,2,30566.0,22022.0,37454.0,178320


In [31]:
# ============================================
# STEP 9.20: RECALL TEST FOR RAREST-TOKEN
# CONFIGURATIONS
# ============================================

def target_is_covered_by_rare_blocking(
    row,
    target_id,
    max_frequency,
    max_tokens_per_field
):
    """
    Check whether a known ground-truth target would
    be retrieved by the rare-token blocking strategy.
    """

    country = row["country_norm"]

    # ----------------------------------------
    # Determine target source
    # ----------------------------------------

    if target_id in s2_lookup:
        target_source = "S2"
        target = s2_lookup[target_id]

        name_freq_index = s2_name_freq
        address_freq_index = s2_address_freq

    elif target_id in s3_lookup:
        target_source = "S3"
        target = s3_lookup[target_id]

        name_freq_index = s3_name_freq
        address_freq_index = s3_address_freq

    else:
        return False

    # Country condition
    if country != target["country_norm"]:
        return False

    # ----------------------------------------
    # Source-1 tokens
    # ----------------------------------------

    source_name_tokens = set(
        get_name_tokens(row["name_norm"])
    )

    source_address_tokens = set(
        get_address_tokens(row["address_norm"])
    )

    # ----------------------------------------
    # Name tokens
    # ----------------------------------------

    name_candidates = []

    for token in source_name_tokens:

        s2_freq = s2_name_freq.get(
            (country, token),
            0
        )

        s3_freq = s3_name_freq.get(
            (country, token),
            0
        )

        frequencies = [
            f for f in [s2_freq, s3_freq]
            if f > 0
        ]

        if frequencies:

            frequency = min(frequencies)

            if frequency <= max_frequency:
                name_candidates.append(
                    (token, frequency)
                )

    name_candidates.sort(
        key=lambda x: x[1]
    )

    selected_name_tokens = {
        token
        for token, _ in
        name_candidates[:max_tokens_per_field]
    }

    # ----------------------------------------
    # Address tokens
    # ----------------------------------------

    address_candidates = []

    for token in source_address_tokens:

        s2_freq = s2_address_freq.get(
            (country, token),
            0
        )

        s3_freq = s3_address_freq.get(
            (country, token),
            0
        )

        frequencies = [
            f for f in [s2_freq, s3_freq]
            if f > 0
        ]

        if frequencies:

            frequency = min(frequencies)

            if frequency <= max_frequency:
                address_candidates.append(
                    (token, frequency)
                )

    address_candidates.sort(
        key=lambda x: x[1]
    )

    selected_address_tokens = {
        token
        for token, _ in
        address_candidates[:max_tokens_per_field]
    }

    # ----------------------------------------
    # Check whether target shares selected
    # name token
    # ----------------------------------------

    target_name_tokens = set(
        get_name_tokens(target["name_norm"])
    )

    for token in selected_name_tokens:

        if token in target_name_tokens:

            target_frequency = name_freq_index.get(
                (country, token),
                0
            )

            if (
                target_frequency > 0
                and target_frequency <= max_frequency
            ):
                return True

    # ----------------------------------------
    # Check whether target shares selected
    # address token
    # ----------------------------------------

    target_address_tokens = set(
        get_address_tokens(target["address_norm"])
    )

    for token in selected_address_tokens:

        if token in target_address_tokens:

            target_frequency = address_freq_index.get(
                (country, token),
                0
            )

            if (
                target_frequency > 0
                and target_frequency <= max_frequency
            ):
                return True

    return False


# --------------------------------------------
# Configurations to evaluate
# --------------------------------------------

configs = [
    (25000, 1),
    (50000, 1),
    (100000, 1),
    (25000, 2),
    (50000, 2),
    (100000, 2)
]


recall_results = []


for max_frequency, n_tokens in configs:

    print(
        f"\nTesting frequency={max_frequency}, "
        f"tokens={n_tokens}"
    )

    total_actual_matches = 0
    covered_matches = 0

    for _, row in sample_gt.iterrows():

        actual_matches = parse_matches(
            row["matched_entity_ids"]
        )

        for target_id in actual_matches:

            total_actual_matches += 1

            if target_is_covered_by_rare_blocking(
                row,
                target_id,
                max_frequency,
                n_tokens
            ):
                covered_matches += 1

    recall = (
        covered_matches /
        total_actual_matches
        if total_actual_matches > 0
        else 0
    )

    recall_results.append({
        "max_frequency": max_frequency,
        "tokens_per_field": n_tokens,
        "total_actual_matches": total_actual_matches,
        "covered_matches": covered_matches,
        "missed_matches": (
            total_actual_matches -
            covered_matches
        ),
        "recall_percent": recall * 100
    })

    print(
        f"Recall: {recall * 100:.2f}%"
    )


recall_df = pd.DataFrame(recall_results)

print("\n" + "=" * 80)
print("RAREST-TOKEN RECALL COMPARISON")
print("=" * 80)

display(
    recall_df.round(2)
)


Testing frequency=25000, tokens=1
Recall: 93.67%

Testing frequency=50000, tokens=1
Recall: 93.96%

Testing frequency=100000, tokens=1
Recall: 93.99%

Testing frequency=25000, tokens=2
Recall: 97.58%

Testing frequency=50000, tokens=2
Recall: 97.76%

Testing frequency=100000, tokens=2
Recall: 97.83%

RAREST-TOKEN RECALL COMPARISON


,max_frequency,tokens_per_field,total_actual_matches,covered_matches,missed_matches,recall_percent
0,25000,1,17383,16283,1100,93.67
1,50000,1,17383,16333,1050,93.96
2,100000,1,17383,16338,1045,93.99
3,25000,2,17383,16962,421,97.58
4,50000,2,17383,16994,389,97.76
5,100000,2,17383,17005,378,97.83


In [32]:
# ============================================
# STEP 9.21: NAME + ADDRESS INTERSECTION
# ============================================

INTERSECTION_FREQUENCY = 100_000


def get_rare_token_candidates(
    row,
    field,
    max_frequency
):
    """
    Retrieve candidates using the single rarest
    eligible token from one field.
    """

    country = row["country_norm"]

    if field == "name":

        tokens = set(
            get_name_tokens(row["name_norm"])
        )

        index_s2 = s2_name_filtered
        index_s3 = s3_name_filtered

        freq_s2 = s2_name_freq
        freq_s3 = s3_name_freq

    else:

        tokens = set(
            get_address_tokens(row["address_norm"])
        )

        index_s2 = s2_address_filtered
        index_s3 = s3_address_filtered

        freq_s2 = s2_address_freq
        freq_s3 = s3_address_freq


    token_candidates = []

    for token in tokens:

        s2_freq = freq_s2.get(
            (country, token),
            0
        )

        s3_freq = freq_s3.get(
            (country, token),
            0
        )

        frequencies = [
            f for f in [s2_freq, s3_freq]
            if f > 0 and f <= max_frequency
        ]

        if frequencies:

            token_candidates.append(
                (token, min(frequencies))
            )


    # Most selective token first
    token_candidates.sort(
        key=lambda x: x[1]
    )


    if not token_candidates:
        return set()


    token = token_candidates[0][0]

    candidates = set()

    candidates.update(
        index_s2.get(
            (country, token),
            []
        )
    )

    candidates.update(
        index_s3.get(
            (country, token),
            []
        )
    )

    return candidates


def generate_intersection_candidates(row):

    # ----------------------------------------
    # Name candidates
    # ----------------------------------------

    name_candidates = get_rare_token_candidates(
        row,
        "name",
        INTERSECTION_FREQUENCY
    )


    # ----------------------------------------
    # Address candidates
    # ----------------------------------------

    address_candidates = get_rare_token_candidates(
        row,
        "address",
        INTERSECTION_FREQUENCY
    )


    # ----------------------------------------
    # Intersection
    # ----------------------------------------

    intersection = (
        name_candidates &
        address_candidates
    )


    return intersection

In [33]:
# ============================================
# STEP 9.22: TEST INTERSECTION VOLUME
# ============================================

test_sample = s1.sample(
    n=100,
    random_state=42
).copy()

candidate_counts = []

for _, row in test_sample.iterrows():

    candidates = generate_intersection_candidates(
        row
    )

    candidate_counts.append(
        len(candidates)
    )


stats = pd.Series(
    candidate_counts
).describe()


print("Intersection candidate generation test complete!")

print("\nCandidate statistics:")
print(stats)

Intersection candidate generation test complete!

Candidate statistics:
count    100.000000
mean       3.030000
std        2.622571
min        0.000000
25%        2.000000
50%        3.000000
75%        4.000000
max       21.000000
dtype: float64


In [34]:
# ============================================
# STEP 9.23: INTERSECTION RECALL
# ============================================

total_actual_matches = 0
covered_matches = 0

for _, row in sample_gt.iterrows():

    actual_matches = parse_matches(
        row["matched_entity_ids"]
    )

    candidates = generate_intersection_candidates(
        row
    )

    total_actual_matches += len(
        actual_matches
    )

    covered_matches += len(
        actual_matches.intersection(
            candidates
        )
    )


recall = (
    covered_matches /
    total_actual_matches
    if total_actual_matches > 0
    else 0
)


print("INTERSECTION BLOCKING RECALL")
print("=" * 50)

print(
    "Total actual matches:",
    total_actual_matches
)

print(
    "Covered matches:",
    covered_matches
)

print(
    "Missed matches:",
    total_actual_matches - covered_matches
)

print(
    f"Recall: {recall * 100:.2f}%"
)

INTERSECTION BLOCKING RECALL
Total actual matches: 17383
Covered matches: 9919
Missed matches: 7464
Recall: 57.06%


In [35]:
# ============================================
# STEP 9.24: ANALYZE MISSED MATCHES
# ============================================

def get_match_blocking_details(row, target_id):

    country = row["country_norm"]

    # ----------------------------------------
    # Find target
    # ----------------------------------------

    if target_id in s2_lookup:
        target = s2_lookup[target_id]
        name_freq_index = s2_name_freq
        address_freq_index = s2_address_freq

    elif target_id in s3_lookup:
        target = s3_lookup[target_id]
        name_freq_index = s3_name_freq
        address_freq_index = s3_address_freq

    else:
        return None

    # ----------------------------------------
    # Source tokens
    # ----------------------------------------

    source_name = set(
        get_name_tokens(row["name_norm"])
    )

    source_address = set(
        get_address_tokens(row["address_norm"])
    )

    target_name = set(
        get_name_tokens(target["name_norm"])
    )

    target_address = set(
        get_address_tokens(target["address_norm"])
    )

    # ----------------------------------------
    # Shared tokens
    # ----------------------------------------

    shared_name = source_name & target_name
    shared_address = source_address & target_address

    # ----------------------------------------
    # Frequencies
    # ----------------------------------------

    name_frequencies = []

    for token in shared_name:

        freq = name_freq_index.get(
            (country, token),
            0
        )

        if freq > 0:
            name_frequencies.append(
                (token, freq)
            )

    address_frequencies = []

    for token in shared_address:

        freq = address_freq_index.get(
            (country, token),
            0
        )

        if freq > 0:
            address_frequencies.append(
                (token, freq)
            )

    return {
        "source1_name": row["name_norm"],
        "target_name": target["name_norm"],
        "source1_address": row["address_norm"],
        "target_address": target["address_norm"],
        "shared_name_tokens": shared_name,
        "shared_address_tokens": shared_address,
        "name_frequencies": name_frequencies,
        "address_frequencies": address_frequencies
    }


# --------------------------------------------
# Find matches missed by 100K + 2-token rule
# --------------------------------------------

missed_examples = []

for _, row in sample_gt.iterrows():

    actual_matches = parse_matches(
        row["matched_entity_ids"]
    )

    for target_id in actual_matches:

        covered = target_is_covered_by_rare_blocking(
            row,
            target_id,
            100000,
            2
        )

        if not covered:

            details = get_match_blocking_details(
                row,
                target_id
            )

            if details:

                details["source1_entity_id"] = (
                    row["entity_id"]
                )

                details["target_entity_id"] = target_id

                missed_examples.append(
                    details
                )

            if len(missed_examples) >= 30:
                break

    if len(missed_examples) >= 30:
        break


missed_df = pd.DataFrame(
    missed_examples
)

print(
    "Number of missed examples collected:",
    len(missed_df)
)

display(missed_df)

Number of missed examples collected: 30


,source1_name,target_name,source1_address,target_address,shared_name_tokens,shared_address_tokens,name_frequencies,address_frequencies,source1_entity_id,target_entity_id
0,creative projects limited,क र एट व प र ज क ट स ल म ट ड,12thfloor c 1204 roya oasis jankalyan nagarmal...,12thfloor 1 mumbai mumbai city मह र ष ट र,{},"{mumbai, city, 12thfloor}",[],"[(mumbai, 197983), (city, 139838), (12thfloor,...",S1-796421498,S3-14934788
1,lakshmi consultants private limited,लक ष म क सल ट ट स प र इव ट ल म ट ड,a 301 new sai dham chsl ramdev park road thane...,a 301 thane मह र ष ट र,{},"{301, thane}",[],"[(301, 8887), (thane, 57375)]",S1-97176033,S3-581985190
2,ss technology private limited,एसएस ट क न ल ज प र इव ट ल म ट ड,shop no 1164 pvt no 2 first floor kucha mahaja...,shop no 01164 delhi central delhi delhi,{},"{delhi, central, shop}",[],"[(delhi, 262208), (central, 11062), (shop, 429...",S1-983540069,S2-4190133
3,ss construction private limited,एसएस क स ट रक शन प र इव ट ल म ट ड,flat no 904 a wing pristine fontana building s...,flat no 904 pune maharashtra,{},"{pune, maharashtra, 904, flat}",[],"[(pune, 78224), (maharashtra, 319026), (904, 1...",S1-630149953,S2-115238959
4,heartland goldman,heartland6oldman com,704 carolina street hoxie ar,704 caroina saint ar walnut ridge,{},{704},[],"[(704, 1250)]",S1-322066165,S2-634792347
5,tech infotech private limited,ట క ఇన ఫ ట క ప ర వ ట ల మ ట డ,sri siddi vinayaka apartm telangana karimnagar...,8 6 375 3 8 karimnagar hyderabad త ల గ ణ,{},"{karimnagar, 375}",[],"[(karimnagar, 778), (375, 878)]",S1-680416224,S2-157721942
6,tech infotech private limited,ట క ఇన ఫ ట క ప ర వ ట ల మ ట డ,sri siddi vinayaka apartm telangana karimnagar...,8 6 375 3 8 karimnagar hyderabad telangana,{},"{telangana, karimnagar, 375}",[],"[(telangana, 75193), (karimnagar, 778), (375, ...",S1-680416224,S2-192452091
7,tech infotech private limited,ట క ఇన ఫ ట క ప ర వ ట ల మ ట డ,sri siddi vinayaka apartm telangana karimnagar...,hyderabad karimnagar త ల గ ణ 8 6 375 3 8,{},"{karimnagar, 375}",[],"[(karimnagar, 778), (375, 878)]",S1-680416224,S2-211421430
8,premier sunrise international private limited,പ ര മ യർ സൺറ സ ഇന റർന ഷണൽ പ ര വറ റ ല മ റ റഡ,house 40 445 mamangalam palarivattom p o ernak...,house ernakulam kerala keralam,{},"{house, kerala, ernakulam}",[],"[(house, 67307), (kerala, 4837), (ernakulam, 2...",S1-867998778,S3-807085228
9,premier sunrise international private limited,പ ര മ യർ സൺറ സ ഇന റർന ഷണൽ പ ര വറ റ ല മ റ റഡ,house 40 445 mamangalam palarivattom p o ernak...,ernakulam kerala house ernakulam n a,{},"{house, kerala, ernakulam}",[],"[(house, 72045), (kerala, 47627), (ernakulam, ...",S1-867998778,S2-126598464


In [36]:
# ============================================
# STEP 9.25: ANALYZE MISSED MATCH PATTERNS
# ============================================

analysis_results = []

for _, row in sample_gt.iterrows():

    actual_matches = parse_matches(
        row["matched_entity_ids"]
    )

    for target_id in actual_matches:

        # Check the 100K + 2-token strategy
        covered = target_is_covered_by_rare_blocking(
            row,
            target_id,
            100000,
            2
        )

        if covered:
            continue

        details = get_match_blocking_details(
            row,
            target_id
        )

        if details is None:
            continue

        shared_name = details[
            "shared_name_tokens"
        ]

        shared_address = details[
            "shared_address_tokens"
        ]

        analysis_results.append({

            "source1_entity_id":
                row["entity_id"],

            "target_entity_id":
                target_id,

            "shared_name_token_count":
                len(shared_name),

            "shared_address_token_count":
                len(shared_address),

            "has_shared_name":
                len(shared_name) > 0,

            "has_shared_address":
                len(shared_address) > 0,

            "name_frequencies":
                details["name_frequencies"],

            "address_frequencies":
                details["address_frequencies"]
        })


missed_analysis = pd.DataFrame(
    analysis_results
)


print(
    "Total missed matches:",
    len(missed_analysis)
)

print("\nMissed-match pattern:")
print("=" * 60)

print(
    "No shared name tokens:",
    (
        ~missed_analysis["has_shared_name"]
    ).sum()
)

print(
    "No shared address tokens:",
    (
        ~missed_analysis["has_shared_address"]
    ).sum()
)

print(
    "No shared name OR address tokens:",
    (
        (~missed_analysis["has_shared_name"]) &
        (~missed_analysis["has_shared_address"])
    ).sum()
)

print(
    "Shared name tokens:",
    missed_analysis["has_shared_name"].sum()
)

print(
    "Shared address tokens:",
    missed_analysis["has_shared_address"].sum()
)

print(
    "Shared both:",
    (
        missed_analysis["has_shared_name"] &
        missed_analysis["has_shared_address"]
    ).sum()
)

Total missed matches: 378

Missed-match pattern:
No shared name tokens: 363
No shared address tokens: 17
No shared name OR address tokens: 10
Shared name tokens: 15
Shared address tokens: 361
Shared both: 8


In [37]:
# ============================================
# STEP 9.25B: SHOW PATTERN PERCENTAGES
# ============================================

total = len(missed_analysis)

if total > 0:

    print(
        f"No shared name: "
        f"{(~missed_analysis['has_shared_name']).sum() / total * 100:.2f}%"
    )

    print(
        f"No shared address: "
        f"{(~missed_analysis['has_shared_address']).sum() / total * 100:.2f}%"
    )

    print(
        f"No shared name OR address: "
        f"{((~missed_analysis['has_shared_name']) & (~missed_analysis['has_shared_address'])).sum() / total * 100:.2f}%"
    )

    print(
        f"Shared both: "
        f"{(missed_analysis['has_shared_name'] & missed_analysis['has_shared_address']).sum() / total * 100:.2f}%"
    )

No shared name: 96.03%
No shared address: 4.50%
No shared name OR address: 2.65%
Shared both: 2.12%


In [38]:
# ============================================
# STEP 9.26: ANALYZE MISSED ADDRESS TOKENS
# ============================================

address_analysis = []

for _, row in sample_gt.iterrows():

    actual_matches = parse_matches(
        row["matched_entity_ids"]
    )

    for target_id in actual_matches:

        covered = target_is_covered_by_rare_blocking(
            row,
            target_id,
            100000,
            2
        )

        if covered:
            continue

        if target_id not in s2_lookup and target_id not in s3_lookup:
            continue

        # ------------------------------------
        # Target information
        # ------------------------------------

        if target_id in s2_lookup:

            target = s2_lookup[target_id]
            freq_index = s2_address_freq

        else:

            target = s3_lookup[target_id]
            freq_index = s3_address_freq


        country = row["country_norm"]

        source_tokens = set(
            get_address_tokens(
                row["address_norm"]
            )
        )

        target_tokens = set(
            get_address_tokens(
                target["address_norm"]
            )
        )

        shared_tokens = (
            source_tokens &
            target_tokens
        )

        if not shared_tokens:
            continue


        # ------------------------------------
        # Frequency of every shared token
        # ------------------------------------

        shared_info = []

        for token in shared_tokens:

            frequency = freq_index.get(
                (country, token),
                0
            )

            shared_info.append(
                (token, frequency)
            )


        shared_info.sort(
            key=lambda x: x[1]
        )


        # ------------------------------------
        # Rank of shared tokens among
        # Source-1 address tokens
        # ------------------------------------

        source_token_info = []

        for token in source_tokens:

            frequency = freq_index.get(
                (country, token),
                0
            )

            if frequency > 0:

                source_token_info.append(
                    (token, frequency)
                )

        source_token_info.sort(
            key=lambda x: x[1]
        )


        source_token_rank = {
            token: rank + 1
            for rank, (token, _) in
            enumerate(source_token_info)
        }


        address_analysis.append({

            "source1_entity_id":
                row["entity_id"],

            "target_entity_id":
                target_id,

            "shared_tokens":
                shared_info,

            "source_token_count":
                len(source_token_info),

            "best_shared_frequency":
                min(
                    freq
                    for _, freq in shared_info
                    if freq > 0
                ),

            "best_shared_token":
                min(
                    shared_info,
                    key=lambda x: x[1]
                )[0],

            "best_shared_rank":
                min(
                    source_token_rank.get(
                        token,
                        999
                    )
                    for token, freq in shared_info
                    if freq > 0
                )
        })


address_analysis_df = pd.DataFrame(
    address_analysis
)


print(
    "Missed matches with shared address tokens:",
    len(address_analysis_df)
)

print("\nBest shared-token frequency:")
print(
    address_analysis_df[
        "best_shared_frequency"
    ].describe()
)

print("\nBest shared-token rank:")
print(
    address_analysis_df[
        "best_shared_rank"
    ].value_counts().sort_index()
)

Missed matches with shared address tokens: 361

Best shared-token frequency:
count       361.000000
mean      27373.193906
std       47194.212344
min           3.000000
25%         910.000000
50%        3405.000000
75%       22999.000000
max      262208.000000
Name: best_shared_frequency, dtype: float64

Best shared-token rank:
best_shared_rank
1       4
2      22
3     134
4      61
5      58
6      36
7      21
8       9
9       5
10      5
11      2
12      3
13      1
Name: count, dtype: int64


In [39]:
# ============================================
# STEP 9.26B: CHECK ADDRESS RECOVERY
# ============================================

for threshold in [25000, 50000, 100000, 200000]:

    recovered = (
        address_analysis_df[
            "best_shared_frequency"
        ] <= threshold
    ).sum()

    total = len(address_analysis_df)

    print(
        f"Threshold {threshold:>6}: "
        f"{recovered}/{total} "
        f"({recovered / total * 100:.2f}%)"
    )


print("\nRecovery by token rank:")

for rank in [1, 2, 3, 4, 5]:

    count = (
        address_analysis_df[
            "best_shared_rank"
        ] <= rank
    ).sum()

    total = len(address_analysis_df)

    print(
        f"Top {rank} address tokens: "
        f"{count}/{total} "
        f"({count / total * 100:.2f}%)"
    )

Threshold  25000: 271/361 (75.07%)
Threshold  50000: 287/361 (79.50%)
Threshold 100000: 324/361 (89.75%)
Threshold 200000: 360/361 (99.72%)

Recovery by token rank:
Top 1 address tokens: 4/361 (1.11%)
Top 2 address tokens: 26/361 (7.20%)
Top 3 address tokens: 160/361 (44.32%)
Top 4 address tokens: 221/361 (61.22%)
Top 5 address tokens: 279/361 (77.29%)


In [40]:
# ============================================
# STEP 9.27: ADDRESS TOP-5 BLOCKING
# ============================================

ADDRESS_MAX_FREQUENCY = 200_000
ADDRESS_MAX_TOKENS = 5


def generate_address_topk_candidates(row):

    country = row["country_norm"]

    address_tokens = set(
        get_address_tokens(
            row["address_norm"]
        )
    )

    token_info = []

    for token in address_tokens:

        s2_freq = s2_address_freq.get(
            (country, token),
            0
        )

        s3_freq = s3_address_freq.get(
            (country, token),
            0
        )

        frequencies = [
            f for f in [s2_freq, s3_freq]
            if f > 0
            and f <= ADDRESS_MAX_FREQUENCY
        ]

        if frequencies:

            token_info.append(
                (token, min(frequencies))
            )

    # Most selective tokens first
    token_info.sort(
        key=lambda x: x[1]
    )

    selected_tokens = token_info[
        :ADDRESS_MAX_TOKENS
    ]

    candidates = set()

    for token, _ in selected_tokens:

        candidates.update(
            s2_address_index.get(
                (country, token),
                []
            )
        )

        candidates.update(
            s3_address_index.get(
                (country, token),
                []
            )
        )

    return candidates


# --------------------------------------------
# Candidate volume test
# --------------------------------------------

test_sample = s1.sample(
    n=100,
    random_state=42
).copy()

candidate_counts = []

for _, row in test_sample.iterrows():

    candidates = generate_address_topk_candidates(
        row
    )

    candidate_counts.append(
        len(candidates)
    )


stats = pd.Series(
    candidate_counts
).describe()

print("Address top-5 blocking test complete!")

print("\nCandidate statistics:")
print(stats)

Address top-5 blocking test complete!

Candidate statistics:
count       100.000000
mean      70789.350000
std       91891.144679
min         295.000000
25%        5091.000000
50%       31423.500000
75%      113218.500000
max      395942.000000
dtype: float64


In [41]:
# ============================================
# STEP 9.28: ADDRESS TOP-5 RECALL
# ============================================

total_actual_matches = 0
covered_matches = 0

for _, row in sample_gt.iterrows():

    actual_matches = parse_matches(
        row["matched_entity_ids"]
    )

    candidates = generate_address_topk_candidates(
        row
    )

    total_actual_matches += len(
        actual_matches
    )

    covered_matches += len(
        actual_matches.intersection(
            candidates
        )
    )


recall = (
    covered_matches /
    total_actual_matches
    if total_actual_matches > 0
    else 0
)


print("ADDRESS TOP-5 BLOCKING RECALL")
print("=" * 50)

print(
    "Total actual matches:",
    total_actual_matches
)

print(
    "Covered matches:",
    covered_matches
)

print(
    "Missed matches:",
    total_actual_matches - covered_matches
)

print(
    f"Recall: {recall * 100:.2f}%"
)

ADDRESS TOP-5 BLOCKING RECALL
Total actual matches: 17383
Covered matches: 16339
Missed matches: 1044
Recall: 93.99%


In [42]:
# ============================================
# STEP 9.29: ADDRESS MULTI-TOKEN BLOCKING
# ============================================

ADDRESS_TOKEN_FREQUENCY = 200_000
MIN_SHARED_ADDRESS_TOKENS = 2


def generate_address_multitoken_candidates(row):

    country = row["country_norm"]

    source_tokens = set(
        get_address_tokens(
            row["address_norm"]
        )
    )

    # ----------------------------------------
    # Get eligible address tokens
    # ----------------------------------------

    eligible_tokens = []

    for token in source_tokens:

        s2_freq = s2_address_freq.get(
            (country, token),
            0
        )

        s3_freq = s3_address_freq.get(
            (country, token),
            0
        )

        # Keep token if at least one source
        # has a manageable block
        frequencies = [
            f for f in [s2_freq, s3_freq]
            if f > 0 and f <= ADDRESS_TOKEN_FREQUENCY
        ]

        if frequencies:
            eligible_tokens.append(token)

    # ----------------------------------------
    # Retrieve candidates from each token
    # ----------------------------------------

    candidate_token_counts = {}

    for token in eligible_tokens:

        for entity_id in s2_address_index.get(
            (country, token),
            []
        ):
            candidate_token_counts[entity_id] = (
                candidate_token_counts.get(entity_id, 0) + 1
            )

        for entity_id in s3_address_index.get(
            (country, token),
            []
        ):
            candidate_token_counts[entity_id] = (
                candidate_token_counts.get(entity_id, 0) + 1
            )

    # ----------------------------------------
    # Keep candidates sharing >= 2 tokens
    # ----------------------------------------

    candidates = {
        entity_id
        for entity_id, shared_count
        in candidate_token_counts.items()
        if shared_count >= MIN_SHARED_ADDRESS_TOKENS
    }

    return candidates

In [43]:
# ============================================
# STEP 9.30: TEST ADDRESS MULTI-TOKEN VOLUME
# ============================================

test_sample = s1.sample(
    n=100,
    random_state=42
).copy()

candidate_counts = []

for _, row in test_sample.iterrows():

    candidates = generate_address_multitoken_candidates(
        row
    )

    candidate_counts.append(
        len(candidates)
    )


stats = pd.Series(
    candidate_counts
).describe()

print("Address multi-token blocking test complete!")

print("\nCandidate statistics:")
print(stats)

Address multi-token blocking test complete!

Candidate statistics:
count       100.000000
mean      38723.490000
std       69652.050625
min           1.000000
25%          33.000000
50%         897.000000
75%       46624.250000
max      319347.000000
dtype: float64


In [44]:
# ============================================
# STEP 9.31: ADDRESS MULTI-TOKEN RECALL
# ============================================

total_actual_matches = 0
covered_matches = 0

for _, row in sample_gt.iterrows():

    actual_matches = parse_matches(
        row["matched_entity_ids"]
    )

    candidates = generate_address_multitoken_candidates(
        row
    )

    total_actual_matches += len(
        actual_matches
    )

    covered_matches += len(
        actual_matches.intersection(
            candidates
        )
    )


recall = (
    covered_matches / total_actual_matches
    if total_actual_matches > 0
    else 0
)


print("ADDRESS MULTI-TOKEN RECALL")
print("=" * 50)

print(
    "Total actual matches:",
    total_actual_matches
)

print(
    "Covered matches:",
    covered_matches
)

print(
    "Missed matches:",
    total_actual_matches - covered_matches
)

print(
    f"Recall: {recall * 100:.2f}%"
)

ADDRESS MULTI-TOKEN RECALL
Total actual matches: 17383
Covered matches: 15925
Missed matches: 1458
Recall: 91.61%


In [45]:
# ============================================
# STEP 9.32A: BUILD EXACT MATCH INDEXES
# ============================================

from collections import defaultdict

# Exact normalized name + country
s2_exact_name = defaultdict(list)

for _, row in s2.iterrows():
    key = (
        row["country_norm"],
        row["name_norm"]
    )

    if row["name_norm"]:
        s2_exact_name[key].append(
            row["entity_id"]
        )


s3_exact_name = defaultdict(list)

for _, row in s3.iterrows():
    key = (
        row["country_norm"],
        row["name_norm"]
    )

    if row["name_norm"]:
        s3_exact_name[key].append(
            row["entity_id"]
        )


# Exact normalized address + country
s2_exact_address = defaultdict(list)

for _, row in s2.iterrows():
    key = (
        row["country_norm"],
        row["address_norm"]
    )

    if row["address_norm"]:
        s2_exact_address[key].append(
            row["entity_id"]
        )


s3_exact_address = defaultdict(list)

for _, row in s3.iterrows():
    key = (
        row["country_norm"],
        row["address_norm"]
    )

    if row["address_norm"]:
        s3_exact_address[key].append(
            row["entity_id"]
        )


print("Exact indexes created!")

print("S2 exact name blocks:", len(s2_exact_name))
print("S3 exact name blocks:", len(s3_exact_name))
print("S2 exact address blocks:", len(s2_exact_address))
print("S3 exact address blocks:", len(s3_exact_address))

Exact indexes created!
S2 exact name blocks: 3971627
S3 exact name blocks: 4223574
S2 exact address blocks: 4105882
S3 exact address blocks: 4452385


In [46]:
# ============================================
# STEP 9.33: FINAL CANDIDATE GENERATOR
# ============================================

MAX_CANDIDATES_PER_S1 = 500


def generate_final_candidates(row):

    country = row["country_norm"]
    s1_id = row["entity_id"]

    # candidate_id -> evidence
    candidate_info = {}

    # ----------------------------------------
    # Helper
    # ----------------------------------------

    def add_candidate(
        entity_id,
        source,
        exact_name=0,
        exact_address=0,
        shared_address_tokens=0
    ):

        key = (source, entity_id)

        if key not in candidate_info:

            candidate_info[key] = {
                "source1_entity_id": s1_id,
                "candidate_entity_id": entity_id,
                "candidate_source": source,
                "exact_name": exact_name,
                "exact_address": exact_address,
                "shared_address_tokens":
                    shared_address_tokens
            }

        else:

            candidate_info[key][
                "exact_name"
            ] = max(
                candidate_info[key]["exact_name"],
                exact_name
            )

            candidate_info[key][
                "exact_address"
            ] = max(
                candidate_info[key]["exact_address"],
                exact_address
            )

            candidate_info[key][
                "shared_address_tokens"
            ] = max(
                candidate_info[key][
                    "shared_address_tokens"
                ],
                shared_address_tokens
            )


    # ========================================
    # PASS 1: EXACT NAME
    # ========================================

    if row["name_norm"]:

        key = (
            country,
            row["name_norm"]
        )

        for entity_id in s2_exact_name.get(
            key, []
        ):

            add_candidate(
                entity_id,
                "S2",
                exact_name=1
            )

        for entity_id in s3_exact_name.get(
            key, []
        ):

            add_candidate(
                entity_id,
                "S3",
                exact_name=1
            )


    # ========================================
    # PASS 2: EXACT ADDRESS
    # ========================================

    if row["address_norm"]:

        key = (
            country,
            row["address_norm"]
        )

        for entity_id in s2_exact_address.get(
            key, []
        ):

            add_candidate(
                entity_id,
                "S2",
                exact_address=1
            )

        for entity_id in s3_exact_address.get(
            key, []
        ):

            add_candidate(
                entity_id,
                "S3",
                exact_address=1
            )


    # ========================================
    # PASS 3: ADDRESS MULTI-TOKEN
    # ========================================

    address_tokens = set(
        get_address_tokens(
            row["address_norm"]
        )
    )

    token_counts = defaultdict(
        int
    )

    for token in address_tokens:

        # Ignore extremely large blocks
        s2_freq = s2_address_freq.get(
            (country, token),
            0
        )

        s3_freq = s3_address_freq.get(
            (country, token),
            0
        )

        if (
            s2_freq == 0
            and s3_freq == 0
        ):
            continue

        if (
            s2_freq > ADDRESS_TOKEN_FREQUENCY
            and s3_freq > ADDRESS_TOKEN_FREQUENCY
        ):
            continue


        # S2
        for entity_id in s2_address_index.get(
            (country, token),
            []
        ):

            token_counts[
                ("S2", entity_id)
            ] += 1


        # S3
        for entity_id in s3_address_index.get(
            (country, token),
            []
        ):

            token_counts[
                ("S3", entity_id)
            ] += 1


    # Only candidates sharing >= 2 address tokens
    for (source, entity_id), count in token_counts.items():

        if count >= MIN_SHARED_ADDRESS_TOKENS:

            add_candidate(
                entity_id,
                source,
                shared_address_tokens=count
            )


    # ========================================
    # RANK CANDIDATES
    # ========================================

    candidates = list(
        candidate_info.values()
    )

    candidates.sort(
        key=lambda x: (
            x["exact_name"],
            x["exact_address"],
            x["shared_address_tokens"]
        ),
        reverse=True
    )


    # ========================================
    # LIMIT
    # ========================================

    candidates = candidates[
        :MAX_CANDIDATES_PER_S1
    ]

    return candidates

In [47]:
# ============================================
# STEP 9.34: TEST FINAL CANDIDATE GENERATOR
# ============================================

test_sample = s1.sample(
    n=100,
    random_state=42
).copy()

all_test_candidates = []

for _, row in test_sample.iterrows():

    candidates = generate_final_candidates(
        row
    )

    all_test_candidates.extend(
        candidates
    )


test_candidates_df = pd.DataFrame(
    all_test_candidates
)

print("Final candidate-generation test complete!")

print(
    "\nTotal candidate pairs:",
    len(test_candidates_df)
)

print(
    "Unique Source-1 records:",
    test_candidates_df[
        "source1_entity_id"
    ].nunique()
)

print(
    "Average candidates per S1:",
    round(
        len(test_candidates_df) /
        test_sample["entity_id"].nunique(),
        2
    )
)

print("\nCandidate source distribution:")

print(
    test_candidates_df[
        "candidate_source"
    ].value_counts()
)

print("\nSample candidates:")

display(
    test_candidates_df.head(10)
)

Final candidate-generation test complete!

Total candidate pairs: 30309
Unique Source-1 records: 100
Average candidates per S1: 303.09

Candidate source distribution:
candidate_source
S2    23318
S3     6991
Name: count, dtype: int64

Sample candidates:


,source1_entity_id,candidate_entity_id,candidate_source,exact_name,exact_address,shared_address_tokens
0,S1-53356671,S2-224665641,S2,1,0,0
1,S1-53356671,S2-444730103,S2,1,0,0
2,S1-53356671,S3-468658814,S3,1,0,0
3,S1-53356671,S3-104607810,S3,1,0,0
4,S1-53356671,S3-696248950,S3,1,0,0
5,S1-53356671,S2-904412198,S2,0,0,2
6,S1-53356671,S2-453334141,S2,0,0,2
7,S1-53356671,S3-996978203,S3,0,0,2
8,S1-320151505,S2-951553277,S2,0,1,2
9,S1-320151505,S3-975397820,S3,0,0,2


In [49]:
# ============================================
# STEP 9.35: OPTIMIZED FINAL BLOCKING RECALL
# ============================================

def check_candidate_fast(row, target_id):
    """
    Check whether a known ground-truth target would
    be retrieved by the final blocking strategy.

    This avoids generating the full candidate set.
    """

    country = row["country_norm"]

    # ----------------------------------------
    # Identify target source
    # ----------------------------------------

    if target_id in s2_lookup:

        target_source = "S2"
        target = s2_lookup[target_id]

        target_name_freq = s2_name_freq
        target_address_freq = s2_address_freq

    elif target_id in s3_lookup:

        target_source = "S3"
        target = s3_lookup[target_id]

        target_name_freq = s3_name_freq
        target_address_freq = s3_address_freq

    else:
        return False

    # ----------------------------------------
    # Country condition
    # ----------------------------------------

    if country != target["country_norm"]:
        return False

    # ========================================
    # PASS 1: EXACT NAME
    # ========================================

    if (
        row["name_norm"]
        and row["name_norm"] == target["name_norm"]
    ):
        return True

    # ========================================
    # PASS 2: EXACT ADDRESS
    # ========================================

    if (
        row["address_norm"]
        and row["address_norm"] == target["address_norm"]
    ):
        return True

    # ========================================
    # PASS 3: ADDRESS MULTI-TOKEN
    # ========================================

    source_address_tokens = set(
        get_address_tokens(
            row["address_norm"]
        )
    )

    target_address_tokens = set(
        get_address_tokens(
            target["address_norm"]
        )
    )

    shared_tokens = (
        source_address_tokens &
        target_address_tokens
    )

    # Count only tokens whose block is
    # within our allowed frequency.
    valid_shared_count = 0

    for token in shared_tokens:

        frequency = target_address_freq.get(
            (country, token),
            0
        )

        if (
            frequency > 0
            and frequency <= ADDRESS_TOKEN_FREQUENCY
        ):
            valid_shared_count += 1

    if valid_shared_count >= MIN_SHARED_ADDRESS_TOKENS:
        return True

    return False


# ============================================
# RUN FAST RECALL TEST
# ============================================

total_actual_matches = 0
covered_matches = 0

print("Starting optimized recall test...")

for i, (_, row) in enumerate(
    sample_gt.iterrows()
):

    actual_matches = parse_matches(
        row["matched_entity_ids"]
    )

    total_actual_matches += len(
        actual_matches
    )

    for target_id in actual_matches:

        if check_candidate_fast(
            row,
            target_id
        ):
            covered_matches += 1

    if (i + 1) % 500 == 0:
        print(
            f"Processed {i + 1}/"
            f"{len(sample_gt)} records..."
        )


recall = (
    covered_matches /
    total_actual_matches
    if total_actual_matches > 0
    else 0
)


print("\n" + "=" * 60)
print("OPTIMIZED FINAL BLOCKING RECALL")
print("=" * 60)

print(
    "Total actual matches:",
    total_actual_matches
)

print(
    "Covered matches:",
    covered_matches
)

print(
    "Missed matches:",
    total_actual_matches -
    covered_matches
)

print(
    f"Recall: {recall * 100:.2f}%"
)

Starting optimized recall test...
Processed 500/5000 records...
Processed 1000/5000 records...
Processed 1500/5000 records...
Processed 2000/5000 records...
Processed 2500/5000 records...
Processed 3000/5000 records...
Processed 3500/5000 records...
Processed 4000/5000 records...
Processed 4500/5000 records...
Processed 5000/5000 records...

OPTIMIZED FINAL BLOCKING RECALL
Total actual matches: 17383
Covered matches: 16306
Missed matches: 1077
Recall: 93.80%


In [50]:
# ============================================
# STEP 9.36A: PREPARE CANDIDATE OUTPUT
# ============================================

import os
import time

CANDIDATE_FILE = os.path.join(
    OUTPUT_DIR,
    "candidate_pairs.tsv"
)

# Remove previous incomplete output if it exists
if os.path.exists(CANDIDATE_FILE):
    os.remove(CANDIDATE_FILE)

print("Candidate output file:", CANDIDATE_FILE)
print("Existing file removed:", not os.path.exists(CANDIDATE_FILE))

Candidate output file: C:\Users\anupr\Amazon ML Challenge\Output\candidate_pairs.tsv
Existing file removed: True


In [51]:
# ============================================
# STEP 9.37: FULL CHUNKED CANDIDATE GENERATION
# ============================================

CHUNK_SIZE = 1000

total_s1 = len(s1)

total_candidates = 0
total_processed = 0

start_time = time.time()

print("=" * 70)
print("STARTING FULL CANDIDATE GENERATION")
print("=" * 70)

print("Total Source-1 records:", f"{total_s1:,}")
print("Chunk size:", CHUNK_SIZE)
print("Max candidates per S1:", MAX_CANDIDATES_PER_S1)
print("Output:", CANDIDATE_FILE)
print()


# --------------------------------------------
# Process Source 1 in chunks
# --------------------------------------------

for start in range(0, total_s1, CHUNK_SIZE):

    end = min(
        start + CHUNK_SIZE,
        total_s1
    )

    chunk = s1.iloc[
        start:end
    ]

    chunk_candidates = []

    # ----------------------------------------
    # Generate candidates for each S1 record
    # ----------------------------------------

    for _, row in chunk.iterrows():

        candidates = generate_final_candidates(
            row
        )

        chunk_candidates.extend(
            candidates
        )

    # ----------------------------------------
    # Write chunk to disk
    # ----------------------------------------

    if chunk_candidates:

        chunk_df = pd.DataFrame(
            chunk_candidates
        )

        # First chunk writes header
        write_header = (
            start == 0
        )

        chunk_df.to_csv(
            CANDIDATE_FILE,
            sep="\t",
            index=False,
            mode="w" if write_header else "a",
            header=write_header
        )

        total_candidates += len(
            chunk_df
        )

        del chunk_df

    total_processed = end

    # ----------------------------------------
    # Progress
    # ----------------------------------------

    elapsed = time.time() - start_time

    rate = (
        total_processed / elapsed
        if elapsed > 0
        else 0
    )

    remaining = (
        total_s1 - total_processed
    )

    estimated_seconds = (
        remaining / rate
        if rate > 0
        else 0
    )

    print(
        f"Processed "
        f"{total_processed:,}/{total_s1:,} "
        f"({total_processed / total_s1 * 100:.2f}%)"
    )

    print(
        f"Candidates written: "
        f"{total_candidates:,}"
    )

    print(
        f"Rate: "
        f"{rate:.2f} S1 records/sec"
    )

    print(
        f"Estimated remaining: "
        f"{estimated_seconds / 60:.1f} min"
    )

    print("-" * 70)


# --------------------------------------------
# Final statistics
# --------------------------------------------

elapsed = time.time() - start_time

print("\n" + "=" * 70)
print("CANDIDATE GENERATION COMPLETE")
print("=" * 70)

print(
    "Source-1 records processed:",
    f"{total_processed:,}"
)

print(
    "Total candidate pairs:",
    f"{total_candidates:,}"
)

print(
    "Average candidates per S1:",
    f"{total_candidates / total_s1:.2f}"
)

print(
    "Elapsed time:",
    f"{elapsed / 60:.2f} minutes"
)

print(
    "Output file:",
    CANDIDATE_FILE
)


STARTING FULL CANDIDATE GENERATION
Total Source-1 records: 2,206,821
Chunk size: 1000
Max candidates per S1: 500
Output: C:\Users\anupr\Amazon ML Challenge\Output\candidate_pairs.tsv



KeyboardInterrupt: 